<hr style="border: 2px solid #7C6F64; margin-top: 10px;">

# Multi-Digit Image Recognition - CRNN + CTC

Task: membaca bilangan multi-digit dari citra grayscale berisi 1 sampai 5 digit horizontal. Notebook ini memakai **CRNN + CTC Loss** dari scratch, karena output berupa sequence digit dengan panjang variabel. Pendekatan ini lebih cocok daripada klasifikasi langsung ke seluruh angka atau segmentasi manual per digit.


## Kenapa CRNN + CTC?

- Gambar `train/` dan `test/` punya tinggi tetap 32 px, lebar diskret 64/96/128/160 px.
- Label adalah string digit dengan panjang 1 sampai 5.
- Digit bisa overlap, jadi segmentasi satu digit satu kotak berisiko rapuh.
- CTC memungkinkan model belajar alignment kiri-ke-kanan tanpa bounding box digit.

Folder `digits/` tetap dipakai untuk EDA dan opsi eksplorasi/pretraining, tetapi baseline utama dilatih langsung dari `train.csv` karena 50k citra multi-digit sudah cukup besar.


## Instalasi Modul


In [ ]:
%pip install -q scikit-learn matplotlib seaborn pillow opencv-python tqdm

# Biasanya Kaggle/Colab sudah punya torch. Jangan install ulang torch kecuali benar-benar perlu.


## Pustaka


In [ ]:
import os, random, warnings, math, json
from pathlib import Path
from collections import Counter, defaultdict

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageOps
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")


## Seed


In [ ]:
def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seed_everything(42)
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print(torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))


## Settings / Config


In [ ]:
class Settings:
    SEED = 42
    RUN_NAME = "multi_digit_crnn_ctc"

    # Ubah path ini sesuai environment: local, Kaggle, atau Colab.
    DATA_ROOT = Path("/home/majiidkun/Majiid/Competition/GEMASTIK/Seleksi/Image/internal-gemastik-xix-itb-image-recognition")
    TRAIN_PATH = DATA_ROOT / "train"
    TEST_PATH = DATA_ROOT / "test"
    DIGITS_PATH = DATA_ROOT / "digits"
    TRAIN_CSV_PATH = DATA_ROOT / "train.csv"
    SAMPLE_SUBMISSION_PATH = DATA_ROOT / "sample_submission.csv"
    OUTPUT_PATH = Path("/kaggle/working") if Path("/kaggle/working").exists() else DATA_ROOT / "outputs"

    MODEL_REGISTRY = {
        "tiny_crnn": {
            "channels": [32, 64, 128, 128, 256],
            "rnn_hidden": 128,
            "dropout": 0.10,
            "batch_size": 256,
            "lr": 1e-3,
            "description": "CRNN kecil dari scratch, cepat untuk sanity check.",
            "when_to_use": "Pakai jika ingin iterasi cepat atau GPU terbatas.",
        },
        "small_crnn": {
            "channels": [64, 128, 256, 256, 512],
            "rnn_hidden": 256,
            "dropout": 0.15,
            "batch_size": 192,
            "lr": 8e-4,
            "description": "CRNN balanced, default utama untuk dataset 50k.",
            "when_to_use": "Pakai sebagai baseline utama: cukup kuat tapi masih ringan.",
        },
        "wide_crnn": {
            "channels": [64, 128, 256, 512, 512],
            "rnn_hidden": 384,
            "dropout": 0.20,
            "batch_size": 128,
            "lr": 6e-4,
            "description": "CRNN lebih lebar untuk mengejar skor setelah baseline stabil.",
            "when_to_use": "Pakai jika small_crnn underfit dan GPU masih cukup.",
        },
    }

    MODEL_NAME = "small_crnn"
    MODEL_CONFIG = MODEL_REGISTRY[MODEL_NAME]
    OUTPUT_DIR = OUTPUT_PATH / f"{RUN_NAME}_{MODEL_NAME}"

    IMG_HEIGHT = 32
    IMG_WIDTH = 160
    MAX_LABEL_LEN = 5
    NUM_CLASSES = 11  # 10 digit + blank CTC
    BLANK_IDX = 10

    BATCH_SIZE = MODEL_CONFIG["batch_size"]
    NUM_WORKERS = 2
    EPOCHS = 20
    LR = MODEL_CONFIG["lr"]
    WEIGHT_DECAY = 1e-4
    GRAD_CLIP = 5.0
    USE_AMP = True
    VAL_SIZE = 0.12
    EARLY_STOP_PATIENCE = 5

    # Augmentasi sengaja ringan karena digit sintetis sudah bersih.
    AUGMENT = True
    MAX_SHIFT_X = 3
    MAX_SHIFT_Y = 2
    BRIGHTNESS_JITTER = 0.12


CFG = Settings()
CFG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
seed_everything(CFG.SEED)

model_catalog_df = pd.DataFrame.from_dict(CFG.MODEL_REGISTRY, orient="index")
display(model_catalog_df[["description", "when_to_use", "batch_size", "lr", "rnn_hidden"]])
print("Selected model:", CFG.MODEL_NAME)
print("Data root     :", CFG.DATA_ROOT)
print("Output dir    :", CFG.OUTPUT_DIR)


## Load Data


In [ ]:
train_df = pd.read_csv(CFG.TRAIN_CSV_PATH)
sample_submission = pd.read_csv(CFG.SAMPLE_SUBMISSION_PATH)
train_df["label"] = train_df["label"].astype(str)
train_df["path"] = train_df["id"].map(lambda x: CFG.TRAIN_PATH / f"{x}.png")
sample_submission["path"] = sample_submission["id"].map(lambda x: CFG.TEST_PATH / f"{x}.png")

print("train_df:", train_df.shape)
print("sample_submission:", sample_submission.shape)
print("missing train images:", int((~train_df["path"].map(Path.exists)).sum()))
print("missing test images :", int((~sample_submission["path"].map(Path.exists)).sum()))
display(train_df.head())
display(sample_submission.head())


## EDA Dataset


In [ ]:
IMAGE_SUFFIXES = {".png", ".jpg", ".jpeg", ".bmp", ".webp"}


def list_images(root_dir):
    root_dir = Path(root_dir)
    return sorted([p for p in root_dir.rglob("*") if p.suffix.lower() in IMAGE_SUFFIXES])


def load_gray(path):
    img = Image.open(path)
    img = ImageOps.exif_transpose(img)
    return img.convert("L")


def inspect_images(paths, max_items=None):
    rows = []
    sample = list(paths)[:max_items] if max_items else list(paths)
    for p in tqdm(sample, desc="inspect"):
        img = load_gray(p)
        arr = np.array(img)
        rows.append({
            "path": str(p),
            "filename": p.name,
            "width": img.width,
            "height": img.height,
            "min": int(arr.min()),
            "max": int(arr.max()),
            "mean": float(arr.mean()),
            "std": float(arr.std()),
        })
    return pd.DataFrame(rows)

train_meta = inspect_images(train_df["path"], max_items=3000)
test_meta = inspect_images(sample_submission["path"], max_items=3000)

print("Label length distribution")
display(train_df["label"].str.len().value_counts().sort_index().to_frame("count"))
print("Digit frequency")
display(pd.Series(list("".join(train_df["label"]))).value_counts().sort_index().to_frame("count"))
print("Train image sizes")
display(train_meta.groupby(["width", "height"]).size().sort_values(ascending=False).to_frame("count"))
print("Test image sizes")
display(test_meta.groupby(["width", "height"]).size().sort_values(ascending=False).to_frame("count"))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.countplot(x=train_df["label"].str.len(), ax=axes[0])
axes[0].set_title("Label length")
sns.histplot(train_meta["width"], ax=axes[1], discrete=True)
axes[1].set_title("Train width")
sns.histplot(test_meta["width"], ax=axes[2], discrete=True)
axes[2].set_title("Test width")
plt.tight_layout()
plt.show()


## Visualisasi Sampel


In [ ]:
def show_samples(df, title, n=12):
    sample = df.sample(min(n, len(df)), random_state=CFG.SEED).reset_index(drop=True)
    cols = 4
    rows = math.ceil(len(sample) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(14, 2.5 * rows))
    axes = np.array(axes).reshape(rows, cols)
    for ax in axes.ravel():
        ax.axis("off")
    for i, row in sample.iterrows():
        ax = axes.ravel()[i]
        ax.imshow(load_gray(row.path), cmap="gray")
        label = row.label if "label" in row else "?"
        ax.set_title(f"id={row.id} label={label}")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

show_samples(train_df, "Train samples")

# digits/ adalah referensi MNIST single-digit (.jpg). Dipakai untuk EDA opsional.
digit_rows = []
for d in range(10):
    paths = list_images(CFG.DIGITS_PATH / str(d))[:4]
    for p in paths:
        digit_rows.append({"path": p, "label": str(d)})
print("digits count:", sum(len(list_images(CFG.DIGITS_PATH / str(d))) for d in range(10)))
fig, axes = plt.subplots(10, 4, figsize=(8, 16))
for ax in axes.ravel():
    ax.axis("off")
for i, row in enumerate(digit_rows[:40]):
    ax = axes.ravel()[i]
    ax.imshow(load_gray(row["path"]), cmap="gray")
    ax.set_title(row["label"])
plt.tight_layout()
plt.show()


## Encoding Label dan Preprocessing Gambar


In [ ]:
def encode_label(text):
    return [int(ch) for ch in str(text)]


def decode_ctc(indices):
    result = []
    prev = None
    for idx in indices:
        idx = int(idx)
        if idx != CFG.BLANK_IDX and idx != prev:
            result.append(str(idx))
        prev = idx
    return "".join(result)


def preprocess_image(path, augment=False):
    img = load_gray(path)
    arr = np.array(img).astype(np.float32) / 255.0

    if augment:
        if random.random() < 0.6:
            tx = random.randint(-CFG.MAX_SHIFT_X, CFG.MAX_SHIFT_X)
            ty = random.randint(-CFG.MAX_SHIFT_Y, CFG.MAX_SHIFT_Y)
            mat = np.float32([[1, 0, tx], [0, 1, ty]])
            arr = cv2.warpAffine(arr, mat, (arr.shape[1], arr.shape[0]), borderValue=0.0)
        if random.random() < 0.5:
            factor = 1.0 + random.uniform(-CFG.BRIGHTNESS_JITTER, CFG.BRIGHTNESS_JITTER)
            arr = np.clip(arr * factor, 0, 1)

    h, w = arr.shape
    if h != CFG.IMG_HEIGHT:
        new_w = int(round(w * CFG.IMG_HEIGHT / h))
        arr = cv2.resize(arr, (new_w, CFG.IMG_HEIGHT), interpolation=cv2.INTER_AREA)
        h, w = arr.shape

    if w > CFG.IMG_WIDTH:
        arr = cv2.resize(arr, (CFG.IMG_WIDTH, CFG.IMG_HEIGHT), interpolation=cv2.INTER_AREA)
        w = CFG.IMG_WIDTH

    canvas = np.zeros((CFG.IMG_HEIGHT, CFG.IMG_WIDTH), dtype=np.float32)
    canvas[:, :w] = arr
    x = torch.from_numpy(canvas).unsqueeze(0)
    x = (x - 0.5) / 0.5
    return x


## Dataset dan DataLoader


In [ ]:
class MultiDigitDataset(Dataset):
    def __init__(self, df, train=False, has_label=True):
        self.df = df.reset_index(drop=True).copy()
        self.train = train
        self.has_label = has_label

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        x = preprocess_image(row.path, augment=self.train and CFG.AUGMENT)
        item = {"image": x, "id": int(row.id), "path": str(row.path)}
        if self.has_label:
            label = encode_label(row.label)
            item["label"] = torch.tensor(label, dtype=torch.long)
            item["label_text"] = str(row.label)
        return item


def ctc_collate(batch):
    images = torch.stack([b["image"] for b in batch])
    ids = torch.tensor([b["id"] for b in batch], dtype=torch.long)
    paths = [b["path"] for b in batch]
    out = {"image": images, "id": ids, "path": paths}
    if "label" in batch[0]:
        labels = [b["label"] for b in batch]
        out["targets"] = torch.cat(labels)
        out["target_lengths"] = torch.tensor([len(x) for x in labels], dtype=torch.long)
        out["label_text"] = [b["label_text"] for b in batch]
    return out

# Holdout split biasa, bukan K-Fold.
train_df["label_len"] = train_df["label"].str.len()
tr_df, va_df = train_test_split(
    train_df,
    test_size=CFG.VAL_SIZE,
    random_state=CFG.SEED,
    stratify=train_df["label_len"],
)

train_loader = DataLoader(MultiDigitDataset(tr_df, train=True), batch_size=CFG.BATCH_SIZE, shuffle=True, num_workers=CFG.NUM_WORKERS, collate_fn=ctc_collate, pin_memory=True)
val_loader = DataLoader(MultiDigitDataset(va_df, train=False), batch_size=CFG.BATCH_SIZE * 2, shuffle=False, num_workers=CFG.NUM_WORKERS, collate_fn=ctc_collate, pin_memory=True)
test_loader = DataLoader(MultiDigitDataset(sample_submission, train=False, has_label=False), batch_size=CFG.BATCH_SIZE * 2, shuffle=False, num_workers=CFG.NUM_WORKERS, collate_fn=ctc_collate, pin_memory=True)

print(len(tr_df), len(va_df), len(sample_submission))


## Model CRNN + CTC


In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.0):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Dropout2d(dropout) if dropout > 0 else nn.Identity(),
        )

    def forward(self, x):
        return self.block(x)


class CRNNCTC(nn.Module):
    def __init__(self, channels, rnn_hidden, num_classes, dropout=0.1):
        super().__init__()
        c1, c2, c3, c4, c5 = channels
        self.cnn = nn.Sequential(
            ConvBlock(1, c1, dropout=0.0),
            nn.MaxPool2d(2, 2),          # 16 x 80
            ConvBlock(c1, c2, dropout=0.0),
            nn.MaxPool2d(2, 2),          # 8 x 40
            ConvBlock(c2, c3, dropout=dropout),
            ConvBlock(c3, c3, dropout=dropout),
            nn.MaxPool2d((2, 1), (2, 1)), # 4 x 40
            ConvBlock(c3, c4, dropout=dropout),
            ConvBlock(c4, c4, dropout=dropout),
            nn.MaxPool2d((2, 1), (2, 1)), # 2 x 40
            nn.Conv2d(c4, c5, kernel_size=(2, 3), padding=(0, 1), bias=False), # 1 x 40
            nn.BatchNorm2d(c5),
            nn.ReLU(inplace=True),
        )
        self.rnn = nn.LSTM(
            input_size=c5,
            hidden_size=rnn_hidden,
            num_layers=2,
            bidirectional=True,
            batch_first=False,
            dropout=dropout,
        )
        self.head = nn.Linear(rnn_hidden * 2, num_classes)

    def forward(self, x):
        feat = self.cnn(x)          # B, C, 1, T
        feat = feat.squeeze(2)      # B, C, T
        feat = feat.permute(2, 0, 1) # T, B, C
        seq, _ = self.rnn(feat)
        logits = self.head(seq)     # T, B, classes
        return logits


model = CRNNCTC(
    channels=CFG.MODEL_CONFIG["channels"],
    rnn_hidden=CFG.MODEL_CONFIG["rnn_hidden"],
    num_classes=CFG.NUM_CLASSES,
    dropout=CFG.MODEL_CONFIG["dropout"],
).to(DEVICE)

with torch.no_grad():
    sample_batch = next(iter(train_loader))["image"][:2].to(DEVICE)
    sample_logits = model(sample_batch)
print("logit shape [T,B,C]:", tuple(sample_logits.shape))
print("parameters:", sum(p.numel() for p in model.parameters()) / 1e6, "M")


## Training Utilities


In [ ]:
def greedy_decode_batch(logits):
    # logits: T, B, C
    pred = logits.detach().argmax(dim=-1).cpu().numpy().T
    return [decode_ctc(seq) for seq in pred]


def exact_match_score(y_true, y_pred):
    return float(np.mean([a == b for a, b in zip(y_true, y_pred)])) if y_true else 0.0


def char_accuracy(y_true, y_pred):
    correct = 0
    total = 0
    for t, p in zip(y_true, y_pred):
        total += max(len(t), len(p))
        correct += sum(a == b for a, b in zip(t, p))
    return correct / max(total, 1)


def run_epoch(model, loader, optimizer=None, scaler=None):
    is_train = optimizer is not None
    model.train(is_train)
    criterion = nn.CTCLoss(blank=CFG.BLANK_IDX, zero_infinity=True)
    total_loss = 0.0
    all_true, all_pred = [], []

    pbar = tqdm(loader, desc="train" if is_train else "valid", leave=False)
    for batch in pbar:
        x = batch["image"].to(DEVICE)
        targets = batch["targets"].to(DEVICE)
        target_lengths = batch["target_lengths"].to(DEVICE)
        y_true = batch["label_text"]

        with torch.set_grad_enabled(is_train):
            with torch.cuda.amp.autocast(enabled=CFG.USE_AMP and DEVICE.type == "cuda"):
                logits = model(x)
                log_probs = F.log_softmax(logits, dim=-1)
                input_lengths = torch.full((x.size(0),), logits.size(0), dtype=torch.long, device=DEVICE)
                loss = criterion(log_probs, targets, input_lengths, target_lengths)

            if is_train:
                optimizer.zero_grad(set_to_none=True)
                if scaler is not None and CFG.USE_AMP:
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.GRAD_CLIP)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.GRAD_CLIP)
                    optimizer.step()

        total_loss += loss.item() * x.size(0)
        y_pred = greedy_decode_batch(logits)
        all_true.extend(y_true)
        all_pred.extend(y_pred)
        pbar.set_postfix(loss=loss.item(), em=exact_match_score(all_true, all_pred))

    return {
        "loss": total_loss / max(len(loader.dataset), 1),
        "exact_match": exact_match_score(all_true, all_pred),
        "char_acc": char_accuracy(all_true, all_pred),
    }


def fit_model(model, train_loader, val_loader):
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.EPOCHS)
    scaler = torch.cuda.amp.GradScaler(enabled=CFG.USE_AMP and DEVICE.type == "cuda")
    best_score = -1
    bad_epochs = 0
    history = []

    for epoch in range(1, CFG.EPOCHS + 1):
        train_log = run_epoch(model, train_loader, optimizer, scaler)
        val_log = run_epoch(model, val_loader)
        scheduler.step()

        row = {"epoch": epoch, **{f"train_{k}": v for k, v in train_log.items()}, **{f"val_{k}": v for k, v in val_log.items()}}
        history.append(row)
        print(row)

        score = val_log["exact_match"]
        if score > best_score:
            best_score = score
            bad_epochs = 0
            torch.save({"model": model.state_dict(), "cfg": CFG.__dict__, "score": best_score}, CFG.OUTPUT_DIR / "best_model.pt")
            print("Saved best:", best_score)
        else:
            bad_epochs += 1
            if bad_epochs >= CFG.EARLY_STOP_PATIENCE:
                print("Early stopping")
                break
    return pd.DataFrame(history)


## Train


In [ ]:
history_df = fit_model(model, train_loader, val_loader)
history_df.tail()


## Post-Train Analysis


In [ ]:
history_df.to_csv(CFG.OUTPUT_DIR / "history.csv", index=False)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, metric in zip(axes, ["loss", "exact_match", "char_acc"]):
    ax.plot(history_df["epoch"], history_df[f"train_{metric}"], label="train")
    ax.plot(history_df["epoch"], history_df[f"val_{metric}"], label="val")
    ax.set_title(metric)
    ax.grid(alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()

ckpt = torch.load(CFG.OUTPUT_DIR / "best_model.pt", map_location=DEVICE)
model.load_state_dict(ckpt["model"])
model.eval()

rows = []
with torch.no_grad():
    for batch in tqdm(val_loader, desc="val predict"):
        logits = model(batch["image"].to(DEVICE))
        preds = greedy_decode_batch(logits)
        for path, true, pred in zip(batch["path"], batch["label_text"], preds):
            rows.append({"path": path, "true": true, "pred": pred, "correct": true == pred, "len": len(true)})
val_pred_df = pd.DataFrame(rows)
val_pred_df.to_csv(CFG.OUTPUT_DIR / "val_predictions.csv", index=False)
display(val_pred_df.groupby("len")["correct"].mean().to_frame("exact_match_by_len"))
display(val_pred_df[~val_pred_df["correct"]].head(30))

wrong = val_pred_df[~val_pred_df["correct"]].head(12)
if len(wrong):
    fig, axes = plt.subplots(math.ceil(len(wrong) / 4), 4, figsize=(14, 3 * math.ceil(len(wrong) / 4)))
    axes = np.array(axes).reshape(-1)
    for ax in axes:
        ax.axis("off")
    for ax, row in zip(axes, wrong.itertuples(index=False)):
        ax.imshow(load_gray(row.path), cmap="gray")
        ax.set_title(f"true={row.true} pred={row.pred}")
    plt.tight_layout()
    plt.show()


## Inference dan Submission


In [ ]:
@torch.no_grad()
def predict_test(model, loader):
    model.eval()
    rows = []
    for batch in tqdm(loader, desc="test predict"):
        logits = model(batch["image"].to(DEVICE))
        preds = greedy_decode_batch(logits)
        for img_id, path, pred in zip(batch["id"].cpu().numpy(), batch["path"], preds):
            rows.append({"id": int(img_id), "filename": Path(path).name, "label": pred if pred else "0"})
    return pd.DataFrame(rows)

ckpt = torch.load(CFG.OUTPUT_DIR / "best_model.pt", map_location=DEVICE)
model.load_state_dict(ckpt["model"])
test_pred_df = predict_test(model, test_loader)
test_pred_df.to_csv(CFG.OUTPUT_DIR / "test_predictions.csv", index=False)

submission = sample_submission[["id"]].merge(test_pred_df[["id", "label"]], on="id", how="left")
submission["label"] = submission["label"].fillna("0").astype(str)
submission.to_csv(CFG.OUTPUT_DIR / "submission.csv", index=False)
print(CFG.OUTPUT_DIR / "submission.csv")
display(submission.head())
